In [2]:
import os
import re

from unstructured.partition.auto import partition
from unstructured.chunking.basic import chunk_elements
from unstructured.chunking.title import chunk_by_title
from unstructured.chunking.base import ChunkingOptions
from unstructured.chunking.base import PreChunker

In [3]:
from unstructured.cleaners.core import (
    clean_ordered_bullets,
    group_broken_paragraphs,
    clean_prefix
)
from unstructured.documents.elements import NarrativeText, ElementMetadata

import chromadb
from chromadb.utils import embedding_functions
import torch

In [4]:
folder_path = "Thue_test"
sentence_tranformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="AITeamVN/Vietnamese_Embedding")

e:\Coding\Lawchat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
chromadb_client = chromadb.PersistentClient(path="vector_database")

In [ ]:
collection = chromadb_client.create_collection(
    name="new_collection", 
    embedding_function=sentence_tranformer_ef, 
    metadata = {
        "hnsw:space": "cosine",
        "hnsw:construction_ef": 200,
        "hnsw:M": 16,
        "hnsw:search_ef": 50,
        "hnsw:num_threads": 1,
        "hnsw:resize_factor": 1.2,
        "hnsw:batch_size": 1000,
        "hnsw:sync_threshold": 1000,    
    }
)


In [7]:
def clean_word(w: str) -> str:
    letters = set('aáàảãạăắằẳẵặâấầẩẫậbcdđeéèẻẽẹêếềểễệfghiíìỉĩịjklmnoóòỏõọôốồổỗộơớờởỡợpqrstuúùủũụưứừửữựvwxyýỳỷỹỵz0123456789')
    new_w = ''
    for letter in w:
        if letter.lower() in letters or letter == '.':
            new_w += letter.lower()
    return new_w


def preprocessing(doc: str) -> str:
    doc = doc.replace('\n', ' ').replace('==', ' ')
    words = doc.split()
    cleaned_words = [clean_word(word) for word in words]
    new_doc = ' '.join(cleaned_words)
    return new_doc

In [8]:
def process_folder(folder_path, collection):
    id_counter = 0

    for filename in os.listdir(folder_path):
        filepath = os.path.join(folder_path, filename)

        # Bỏ qua thư mục hoặc file không tồn tại
        if not os.path.isfile(filepath):
            continue

        cleaned_elements = []

        try:
            elements = partition(
                filename=filepath, 
                strategy="hi_res", 
                include_metadata=True,    
                max_partition=1000, 
                languages=["eng", "vie"],
                split_pdf_page=True, 
                split_pdf_allow_failed=True, 
                split_pdf_concurrency_level=15
            )
        except Exception as e:
            print(f" Error partitioning {filename}: {e}")
            continue

        text_elements = [
            el for el in elements
            if getattr(el, 'category', None) 
            #not in ('Footer', 'Header')
        ]

        for el in text_elements:
            cleaned_text = preprocessing(el.text)

            if len(cleaned_text) > 10:
                metadata = el.metadata if el.metadata else ElementMetadata()
                metadata.page_number = getattr(el.metadata, "page_number", None)
                cleaned_elements.append(NarrativeText(cleaned_text, metadata=metadata))

        chunks = chunk_elements(cleaned_elements, max_characters=200, new_after_n_chars=180, overlap=True)
        print(f"{filename}: {len(chunks)} chunks")

        documents, metadatas, ids = [], [], []

        for chunk in chunks:
            page_number = chunk.metadata.page_number if (chunk.metadata and chunk.metadata.page_number is not None) else "unknown"

            documents.append(chunk.text)
            metadatas.append({
                "chunk_id": str(id_counter),
                "filename": filename,
                "page_number": page_number,
            })
            ids.append(str(id_counter))
            id_counter += 1

        if documents and metadatas and ids:
            try:
                collection.add(
                    documents=documents,
                    metadatas=metadatas,
                    ids=ids,
                )
                print(f"{filename} added to ChromaDB successfully!")
            except Exception as e:
                print(f"Error adding {filename} to collection: {e}")
        else:
            print(f"{filename} skipped because empty!")

    print("Update successfully!")

In [9]:
process_folder(folder_path, collection)

Van-ban-hop-nhat-Luat-Ban-hanh-van-ban-quy-pham-phap-luat.doc: 1594 chunks
Van-ban-hop-nhat-Luat-Ban-hanh-van-ban-quy-pham-phap-luat.doc added to ChromaDB successfully!
Update successfully!


In [ ]:
# Khởi tạo client
chromadb_client = chromadb.PersistentClient(path="vector_database")

# Xóa collection theo tên
collection_name = "new_collection"
chromadb_client.delete_collection(name=collection_name)

print(f"Collection '{collection_name}' đã được xóa thành công.")

In [11]:
for i in range(1,100, 1):
    batch = collection.get(
        include=["documents","metadatas"],
        limit=1,
        offset=i)
    print(batch) 

{'ids': ['1'], 'embeddings': None, 'documents': ['luật ban hành văn bản quy phạm pháp luật số 802015qh13 ngày 22 tháng 6 năm 2015 của quốc hội có hiệu lực kể từ ngày 01 tháng 7 năm 2016 được sửa đổi bổ sung bởi'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'page_number': 'unknown', 'filename': 'Van-ban-hop-nhat-Luat-Ban-hanh-van-ban-quy-pham-phap-luat.doc', 'chunk_id': '1'}]}
{'ids': ['2'], 'embeddings': None, 'documents': ['luật số 632020qh14 ngày 18 tháng 6 năm 2020 của quốc hội sửa đổi bổ sung một số điều của luật ban hành văn bản quy phạm pháp luật có hiệu lực kể từ ngày 01 tháng 01 năm 2021.'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'chunk_id': '2', 'filename': 'Van-ban-hop-nhat-Luat-Ban-hanh-van-ban-quy-pham-phap-luat.doc', 'page_number': 'unknown'}]}
{'ids': ['3'], 'embeddings': None, 'documents': ['căn cứ hiến pháp nước cộng hòa xã hội chủ nghĩa việt nam\n\nquốc hội ban hành luật ban hành văn bả

In [43]:
question = "Chính sách cơ bản về giáo dục và khoa học được quy định ở đâu?"

sentence_tranformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="AITeamVN/Vietnamese_Embedding")
chromadb_client = chromadb.PersistentClient(path="vector_database")
collection = chromadb_client.get_collection(
    name="new_collection",
    embedding_function= sentence_tranformer_ef,
    )

def similarity(question):
    results = collection.query(
    query_texts= question, 
    n_results= 10,
    include=['documents','distances','metadatas'],
    )
    return results

print(f'count= {collection.count()}')
print(similarity(question))


count= 1594
{'ids': [['121', '152', '122', '119', '584', '997', '763', '1107', '9', '213']], 'embeddings': None, 'documents': [['c chính sách cơ bản về tài chính tiền tệ quốc gia ngân sách nhà nước quy định sửa đổi hoặc bãi bỏ các thứ thuế\n\nd chính sách cơ bản về văn hóa giáo dục y tế khoa học công nghệ môi trường', 'trương của đảng chính sách pháp luật của nhà nước.', 'đ quốc phòng an ninh quốc gia\n\ne chính sách dân tộc chính sách tôn giáo của nhà nước', 'và cơ quan khác do quốc hội thành lập', 'dự thảo tổ chức đánh giá tác động của chính sách để báo cáo quốc hội', 'động giới của chính sách.', 'đề nghị xây dựng nghị định trong đó nêu rõ các chính sách đã được chính phủ thông qua trình thủ tướng chính phủ xem xét và ký ban hành.', 'cấp và tài liệu có liên quan đến dự thảo quyết định', 'trong luật này các từ ngữ dưới đây được hiểu như sau', 'tác động của từng chính sách trong đề nghị xây dựng luật pháp lệnh.']], 'uris': None, 'included': ['documents', 'distances', 'metadatas'], 'dat

In [44]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

def top_bm25(question):
    docs = similarity(question)["documents"][0]      # lấy list các đoạn văn
    metas = similarity(question)["metadatas"][0]     # lấy list metadata tương ứng

    documents = [
        Document(page_content=doc, metadata=meta)
        for doc, meta in zip(docs, metas)
    ]

    # Tạo BM25 retriever
    retriever = BM25Retriever.from_documents(documents)

    # Gọi truy vấn
    bm25_results = retriever.invoke(question)[:2]
    
    return bm25_results

#print(bm25_results)

#print(len(bm25_results))
#print(len(documents))
# In kết quả
for r in top_bm25(question):
    print(r.page_content)
    print("----")


c chính sách cơ bản về tài chính tiền tệ quốc gia ngân sách nhà nước quy định sửa đổi hoặc bãi bỏ các thứ thuế

d chính sách cơ bản về văn hóa giáo dục y tế khoa học công nghệ môi trường
----
và cơ quan khác do quốc hội thành lập
----


In [26]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("VietAI/gpt-neo-1.3B-vietnamese-news")
model = AutoModelForCausalLM.from_pretrained("VietAI/gpt-neo-1.3B-vietnamese-news", low_cpu_mem_usage=True)

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
#model.to(device)


def gen(prompt):
    device = torch.device("cpu")
    input_ids = tokenizer(prompt, return_tensors="pt")['input_ids'].to(device)
    
    gen_tokens = model.generate(
            input_ids,
            max_length=500,
            do_sample=True,
            temperature=0.9,
            top_k=20,
        )
    return gen_tokens

In [45]:
def reply(question):
    top_2 = []
    for r in top_bm25(question)[0:2]:
        #print("----")
        #print(r.page_content)

        prompt = f"""Câu hỏi: {question}

            Dữ liệu: {r.page_content}

            Trả lời:"""
        
        gens = gen(prompt)
            
        gen_text = tokenizer.decode(gens[0], skip_special_tokens=True)
        top_2.append((gen_text, r.metadata))
    return top_2

for r in reply(question):
    print(r)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


('Câu hỏi: Chính sách cơ bản về giáo dục và khoa học được quy định ở đâu?\n\n            Dữ liệu: c chính sách cơ bản về tài chính tiền tệ quốc gia ngân sách nhà nước quy định sửa đổi hoặc bãi bỏ các thứ thuế\n\nd chính sách cơ bản về văn hóa giáo dục y tế khoa học công nghệ môi trường\n\n            Trả lời:\nThứ nhất: Chính sách cơ bản về giáo dục:\nThứ nhất, sửa đổi các chính sách về hệ thống giáo dục quốc dân (tiểu học, trung học cơ sở, trung học phổ thông) được thực hiện theo quy định của Luật Giáo dục năm 2019. Giáo dục mầm non và giáo dục phổ thông có nhiều cấp học, bao gồm mầm non, tiểu học, trung học cơ sở, trung học phổ thông.\nCác quy định về hệ thống giáo dục quốc dân được thực hiện theo quy định của Luật Giáo dục năm 2019 về quy định các nội dung giáo dục bắt buộc đối với mọi người học không phân biệt dân tộc, nam nữ, thành phần xã hội, tín ngưỡng, tôn giáo.\nNgoài ra, các quy định khác về hệ thống giáo dục quốc dân được quy định tại Luật Giáo dục năm 2019.\nThứ hai, sửa đ